In [1]:
#MT Adaptation with SmolDoc: SFT on H100

import os
import torch

from datasets import Dataset, DatasetDict
from transformers import AutoModelForCausalLM, AutoTokenizer

from trl import SFTTrainer, SFTConfig

from utils import get_extended_datasets

torch.set_float32_matmul_precision("high")  # helps on H100

# --- CONFIG ---

SMOLDOC_CONFIG = "smoldoc__en_sw"

# Base model for full SFT (change this to other models: llama, gemma, etc.)
SFT_MODEL_NAME = "google/gemma-3-4b-it"

OUTPUT_DIR_SFT = f"checkpoints/sft_{SMOLDOC_CONFIG}"
os.makedirs("checkpoints", exist_ok=True)

print("Config:", SMOLDOC_CONFIG)
print("SFT base model:", SFT_MODEL_NAME)

# --- Load SmolDoc + factuality annotations, pick one config, build text field ---

datasets = get_extended_datasets(save_path="data/smoldoc_datasets", overwrite=False)
ds_full: Dataset = datasets[SMOLDOC_CONFIG]

print(ds_full)

# Split: 90% for training, 10% held-out test
split_1 = ds_full.train_test_split(test_size=0.1, seed=42)
train_eval_ds = split_1["train"]  # This is the 90%
test_ds = split_1["test"]  # This is the 10% held-out

# Further split the 90% into 80% train and 20% eval (0.8 * 0.9 = 0.72 total, 0.2 * 0.9 = 0.18 total)
split_2 = train_eval_ds.train_test_split(test_size=0.2, seed=42)
train_ds = split_2["train"]  # 72% of original data
eval_ds = split_2["test"]    # 18% of original data

print(f"Train size: {len(train_ds)} ({len(train_ds)/len(ds_full)*100:.1f}% of total)")
print(f"Eval size: {len(eval_ds)} ({len(eval_ds)/len(ds_full)*100:.1f}% of total)")
print(f"Test size: {len(test_ds)} ({len(test_ds)/len(ds_full)*100:.1f}% of total)")

Config: smoldoc__en_sw
SFT base model: google/gemma-3-4b-it
📂 Found existing SmolDoc DatasetDict at data/smoldoc_datasets, loading from disk...
📂 Loaded DatasetDict from data/smoldoc_datasets with 102 configs.
Using cached file: data/smoldoc-factuality-ratings.json
Dataset({
    features: ['id', 'sl', 'tl', 'srcs', 'trgs', 'factuality', 'is_src_orig', 'annotator_1_label', 'annotator_1_notes', 'annotator_2_label', 'annotator_2_notes', 'annotator_3_label', 'annotator_3_notes'],
    num_rows: 584
})
Train size: 420 (71.9% of total)
Eval size: 105 (18.0% of total)
Test size: 59 (10.1% of total)


In [2]:
def format_mt_example(srcs, trgs, lang_name="Swahili"):
    """
    Build a single text sequence of the form:

    You are an expert in English to Swahili translation.
    Translate the following English text into Swahili.

    English:
    <src>

    Swahili:
    <tgt>
    """
    src = " ".join(srcs).strip()
    tgt = " ".join(trgs).strip()
    return (
        "You are an expert in English to Swahili translation.\n"
        "Translate the following English text into Swahili.\n\n"
        f"English:\n{src}\n\nSwahili:\n{tgt}"
    )


def add_text_column(batch):
    texts = [
        format_mt_example(srcs, trgs, lang_name="Swahili")
        for srcs, trgs in zip(batch["srcs"], batch["trgs"])
    ]
    return {"text": texts}


def formatting_func(examples):
    # TRL passes a batch as a dict of lists, e.g. {"text": [...], "id": [...]}
    # We just return the text list.
    return examples["text"]


train_ds_fmt = train_ds.map(add_text_column, batched=True)
eval_ds_fmt = eval_ds.map(add_text_column, batched=True)
test_ds_fmt = test_ds.map(add_text_column, batched=True)

print("Example training text:\n", train_ds_fmt[0]["text"][:400])

Example training text:
 You are an expert in English to Swahili translation.
Translate the following English text into Swahili.

English:
The history of Cameroon is long and complex, dating back to the earliest human settlements in the region. The area was first settled by Bantu peoples around 3000 BCE, and it was later conquered by the Kanem-Bornu Empire in the 11th century. In the 15th century, the Portuguese arrived i


In [3]:
# --- Tokenizer ---
tokenizer_sft = AutoTokenizer.from_pretrained(SFT_MODEL_NAME)
if tokenizer_sft.pad_token is None:
    tokenizer_sft.pad_token = tokenizer_sft.eos_token
tokenizer_sft.padding_side = "right"

# --- SFT Configuration ---
NUM_EPOCHS = 3

# H100 is big, so push the batch size
# Safe starting point for a 4B model @ 2k tokens on 80GB:
per_device_bs = 1        # bump to 8 if it fits
grad_accum = 4           # so global batch = 4 * 1 = 4 sequences

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR_SFT,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=per_device_bs,
    per_device_eval_batch_size=per_device_bs,
    gradient_accumulation_steps=grad_accum,
    learning_rate=5e-5,
    max_length=1024,              # Fixed: was max_length
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=200,
    save_total_limit=2,               # Keep only best 2 checkpoints
    load_best_model_at_end=True,      # Load best model at end
    metric_for_best_model="loss",     # Use validation loss as metric
    bf16=True,                        # H100 loves bf16
    packing=False,                    # can try True later for better throughput
    dataloader_num_workers=4,         # use your CPU a bit
    gradient_checkpointing=False,     # you *can* turn this on, but you have VRAM
    optim="adamw_torch_fused",        # good on NVIDIA GPUs
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
)

# --- Initialize Model ---
print("\nLoading model...")
model_sft = AutoModelForCausalLM.from_pretrained(
    SFT_MODEL_NAME,
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
)
model_sft.config.use_cache = False  # Required for gradient checkpointing

print("Model loaded successfully!")

# --- Create Trainer ---
trainer_sft = SFTTrainer(
    model=model_sft,
    processing_class=tokenizer_sft,
    train_dataset=train_ds_fmt,
    eval_dataset=eval_ds_fmt,
    formatting_func=formatting_func,
    args=sft_config,
)

# --- Train ---
print("\n" + "="*50)
print("STARTING TRAINING")
print("="*50 + "\n")

trainer_sft.train()

# --- Save gemme_first model ---
final_dir = os.path.join(OUTPUT_DIR_SFT, "gemme_first")
trainer_sft.save_model(final_dir)
tokenizer_sft.save_pretrained(final_dir)

print(f"\nTraining finished. Model saved to: {final_dir}")


`torch_dtype` is deprecated! Use `dtype` instead!



Loading model...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded successfully!


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.



STARTING TRAINING



OutOfMemoryError: CUDA out of memory. Tried to allocate 1.25 GiB. GPU 0 has a total capacity of 79.18 GiB of which 1.17 GiB is free. Process 29356 has 25.82 GiB memory in use. Process 29789 has 35.82 GiB memory in use. Including non-PyTorch memory, this process has 16.36 GiB memory in use. Of the allocated memory 15.31 GiB is allocated by PyTorch, and 325.60 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
from transformers import pipeline

# Path to the gemme_first trained model
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR_SFT, "gemme_first")

print("\n\n" + "="*50)
print("LOADING MODEL FOR GENERATION-BASED EVALUATION")
print("="*50)

# --- Load Model and Tokenizer for Inference ---
ft_tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_DIR)
if ft_tokenizer.pad_token is None:
    ft_tokenizer.pad_token = ft_tokenizer.eos_token
ft_tokenizer.padding_side = "left"  # Set to left for better generation speed/results

ft_model = AutoModelForCausalLM.from_pretrained(
    CHECKPOINT_DIR,
    torch_dtype=torch.bfloat16,
    device_map="cuda:0",   # single H100
)

pipe = pipeline(
    "text-generation",
    model=ft_model,
    tokenizer=ft_tokenizer,
    device_map="cuda:0",
)

print("\n\n" + "="*50)
print("FINAL TEST SET TRANSLATIONS (Held-out 10% Split)")
print("="*50)

# Collect predictions and references for BLEU calculation
all_predictions = []
all_references = []

for i in range(len(test_ds)):
    example = test_ds[i]
    src_text = " ".join(example["srcs"]).strip()
    ref_text = " ".join(example["trgs"]).strip()

    # Build the prompt pattern for inference (without the target text)
    prompt = (
        "You are an expert in English to Swahili translation.\n"
        "Translate the following English text into Swahili.\n\n"
        f"English:\n{src_text}\n\nSwahili:\n"
    )

    out = pipe(
        prompt,
        max_new_tokens=1024,  # Increased max tokens for full translation
        do_sample=False,
        return_full_text=False,  # Only return the generated part
    )

    # Extract and clean the generated translation
    generated_text = out[0]["generated_text"].split('\n')[0].strip()

    # Store for BLEU calculation
    all_predictions.append(generated_text)
    all_references.append([ref_text])  # BLEU expects list of references

    print(f"\n--- Example {i+1} / {len(test_ds)} ---")
    print(f"SOURCE (EN): {src_text}")
    print(f"REFERENCE (SW): {ref_text}")
    print(f"MODEL OUTPUT (SW): {generated_text}")

# --- Calculate BLEU Score on Test Set ---
import evaluate

bleu_metric = evaluate.load("bleu")
bleu_result = bleu_metric.compute(predictions=all_predictions, references=all_references)

print("\n\n" + "="*50)
print("TEST SET BLEU SCORE")
print("="*50)
print(f"BLEU: {bleu_result['bleu']:.4f}")
print(f"BLEU Precisions: {bleu_result['precisions']}")
print(f"Brevity Penalty: {bleu_result['brevity_penalty']:.4f}")
print(f"Length Ratio: {bleu_result['length_ratio']:.4f}")

# End of inference and evaluation block